In [ ]:
# using pandas for data analysis
import pandas as pd


# load the relevant data for the 10 rules analysis
invoice_line = pd.read_csv("invoice_line.csv")
deliveries = pd.read_csv("deliveries.csv")
customers = pd.read_csv("customers.csv")


In [ ]:
invoice_line.groupby("order_id")["invoice_date"].nunique().value_counts()


invoice_date
1    50
Name: count, dtype: int64

In [ ]:
#Normalize invoice_line data to one row per order_id
invoice_order = (
    invoice_line
    .drop_duplicates(subset=["order_id"])
    [["order_id", "invoice_date"]]
)

In [10]:
#Join datasets
df = invoice_order.merge(
    deliveries,
    on="order_id",
    how="left"
)

In [11]:
#Convert to datetime
df["invoice_date"] = pd.to_datetime(df["invoice_date"])

In [12]:
# the delivery_confirm_date has data quality issues, the date is given in differents formats and I need first to normalize.
df["delivery_confirm_date"] = pd.to_datetime(
    df["delivery_confirm_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [ ]:
#Calculate the date difference
df["days_diff"] = (df["invoice_date"] - df["delivery_confirm_date"]).dt.days

#Categorize the result
df["status"] = "Fulfill 10-day-rule"
df.loc[df["days_diff"] > 10, "status"] = "Late"
df.loc[df["days_diff"] < 0, "status"] = "Negative date-diff"
df.loc[df["days_diff"].isna(), "status"] = "Missing"

df["status"].value_counts()

status
Fulfill 10-day-rule    22
Late                   16
Negative date-diff     12
Name: count, dtype: int64

In [24]:
df["days_diff"]

0       8
1       5
2       6
3      13
4      20
5      14
6    -176
7       7
8       8
9    -227
10     -9
11     22
12    -71
13     40
14     27
15   -261
16    -67
17     20
18      5
19     27
20      4
21      8
22     10
23     15
24      9
25      8
26   -227
27     23
28      6
29     10
30   -139
31      6
32     19
33     16
34      8
35     15
36    -54
37     25
38     10
39      6
40      6
41     10
42   -166
43   -259
44     22
45     76
46      8
47   -143
48      6
49      5
Name: days_diff, dtype: int64

In [31]:
df[df["status"] == "Check"]

,order_id,invoice_date,delivery_confirm_id,delivery_confirm_date,delivery_country,days_diff,vida_10_day_rule,status
6,ORD-2026-483219,2027-02-06,CONF-560965,2027-08-01,Netherlands,-176,Compliant,Check
9,ORD-2027-296460,2027-02-17,CONF-747653,2027-10-02,France,-227,Compliant,Check
10,ORD-2027-340184,2027-02-21,CONF-831485,2027-03-02,Sweden,-9,Compliant,Check
12,ORD-2027-881543,2027-03-24,CONF-699281,2027-06-03,Germany,-71,Compliant,Check
15,ORD-2027-126344,2027-03-17,CONF-381172,2027-12-03,France,-261,Compliant,Check
16,ORD-2026-339139,2027-02-24,CONF-871248,2027-05-02,France,-67,Compliant,Check
26,ORD-2027-966069,2027-02-17,CONF-941444,2027-10-02,Germany,-227,Compliant,Check
30,ORD-2027-599775,2027-03-17,CONF-260684,2027-08-03,Belgium,-139,Compliant,Check
36,ORD-2027-116776,2027-03-10,CONF-198879,2027-05-03,Netherlands,-54,Compliant,Check
42,ORD-2027-346967,2027-02-17,CONF-526176,2027-08-02,Netherlands,-166,Compliant,Check


In [27]:
df["days_diff"].value_counts().sort_index()

days_diff
-261    1
-259    1
-227    2
-176    1
-166    1
-143    1
-139    1
-71     1
-67     1
-54     1
-9      1
 4      1
 5      3
 6      6
 7      1
 8      6
 9      1
 10     4
 13     1
 14     1
 15     2
 16     1
 19     1
 20     2
 22     2
 23     1
 25     1
 27     2
 40     1
 76     1
Name: count, dtype: int64

In [15]:
# Here are we "Grouping By: count()" as we know from SQL, it counts the invoices that where send late and the ones that where on time
df["vida_10_day_rule"] = df["days_diff"].apply(
    lambda x: "Compliant" if x <= 10 else "Late"
)

In [16]:
#Just showing the results
df["vida_10_day_rule"].value_counts()

vida_10_day_rule
Compliant    34
Late         16
Name: count, dtype: int64